# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook serves as an interactive template for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a FAIR Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets by @id
print("Available record sets:")
record_set_ids = [recset['@id'] for recset in metadata._data.get('recordSet', [])]
for rs in record_set_ids:
    print(f"- {rs}")

# For each record set, print its fields and ids
for rs in metadata._data.get('recordSet', []):
    print(f"\nRecord set: {rs.get('@id')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for f in fields:
        if isinstance(f, dict) and '@id' in f:
            print(f"    - {f['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# For this dataset, the main data is likely in the first record set. Update this to the actual '@id' from above.
record_sets = record_set_ids  # Use all discovered record sets
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records for record set {record_set}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading record set {record_set}: {e}")

# Example: display head for the first record set
main_record_set_id = record_sets[0] if len(record_sets) > 0 else None
if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. You may choose a numeric field and group field from the ones listed above.

In [ ]:
# Choose a numeric field and a grouping field using their column names / @ids. Adjust as per actual field list in the dataset.
if main_record_set_id and len(dataframes) > 0:
    main_df = dataframes[main_record_set_id]
    print(f"Available fields: {main_df.columns.tolist()}")

    # Example: Try common proteomics fields
    import numpy as np
    numeric_fields = [col for col in main_df.columns if main_df[col].dtype in [np.float64, np.int64]]
    if not numeric_fields:
        # Try inferring numeric from non-numeric columns
        for col in main_df.columns:
            try:
                main_df[col] = pd.to_numeric(main_df[col])
                numeric_fields.append(col)
            except:
                continue

    print(f"Numeric fields detected: {numeric_fields}")
    if len(numeric_fields) == 0:
        print("No numeric fields detected.")
    else:
        numeric_field = numeric_fields[0]
        # Filter rows with the numeric field > threshold
        threshold = main_df[numeric_field].mean() if not pd.isnull(main_df[numeric_field].mean()) else 10
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first non-numeric field
        non_numeric_fields = [c for c in main_df.columns if c not in numeric_fields]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped average of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot histogram and boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(dataframes) > 0:
    df = dataframes[main_record_set_id]
    # Try plotting the first numeric field
    if 'numeric_field' in locals():
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f'Histogram of {numeric_field}')

        plt.subplot(1, 2, 2)
        sns.boxplot(x=df[numeric_field].dropna())
        plt.title(f'Boxplot of {numeric_field}')
        plt.show()
    else:
        print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we loaded Mass Spectrometry extracellular vesicle data using `mlcroissant`, reviewed available record sets and their fields, and conducted a basic exploratory data analysis and visualization. Further domain-specific analyses can be performed by selecting additional fields and applying relevant proteomics data science workflows.